In [11]:
# === SESSION BOOTSTRAP ===  (SWITCH RUNTIME TO GPU; this is a ~1 hour GPU run: 10 seeds x 5 families x 20 epochs)
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT="/content/drive/MyDrive/UAV_TRUST_Research"; REPO=f"{PARENT}/uav-trust-research"
for fn in (".gitconfig",".git-credentials"):
    p=os.path.join(PARENT,fn)
    if os.path.exists(p): subprocess.run(f'cp "{p}" /root/{fn}',shell=True)
subprocess.run("git config --global credential.helper store",shell=True)
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0,REPO) if REPO not in sys.path else None; print("cwd:",os.getcwd())
else: print("run 00_setup first")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [12]:
!pip install pandas numpy pyarrow scikit-learn matplotlib --quiet
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

torch 2.11.0+cu128 | CUDA available: True


In [13]:
# Representation-invariance test (REFINED): 1D-CNN on raw packet sequences of the SAME flows,
# 10 seeds, longer schedule, class-and-family-balanced sampling. Conformal coverage panel only.
CAS_TS_PARQUET="/content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research/data/uav_cas/uav_cas_ts.parquet"
CAS_STAT_CMP="reports/14_uavcas_panel_raw.csv"
NORMAL_KEY="benign"; L=1105; SUBSAMPLE_BENIGN=20000
CFG={"seeds":list(range(10)),"alpha":0.10,"epochs":20,"batch":256,"lr":1e-3,"weight_decay":1e-5,
     "normal_fracs":{"train":0.60,"cal":0.20,"test_seen":0.10,"test_shift":0.10},
     "family_fracs":{"train":0.60,"cal":0.20,"test_seen":0.20},
     "fig_dir":"figures","report_dir":"reports"}
import os
for d in [CFG["fig_dir"],CFG["report_dir"]]: os.makedirs(d,exist_ok=True)
print("configured; L =",L,"| seeds",len(CFG["seeds"]),"| epochs",CFG["epochs"])

configured; L = 1105 | seeds 10 | epochs 20


In [14]:
import numpy as np, pandas as pd, gc, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from src.trust import conformal_qhat
dev="cuda" if torch.cuda.is_available() else "cpu"; print("device:",dev)
if dev=="cpu": print("WARNING: no GPU detected -> switch runtime to GPU, this will be very slow on CPU.")

ts=pd.read_parquet(CAS_TS_PARQUET); print("loaded ts flows:",len(ts))
lab=ts["Label"].astype(str); nv=[v for v in ts["Label"].unique() if str(v).lower()==NORMAL_KEY][0]
rng=np.random.default_rng(42)
idx_norm=np.where(lab.values==nv)[0]
if len(idx_norm)>SUBSAMPLE_BENIGN: idx_norm=rng.choice(idx_norm,SUBSAMPLE_BENIGN,replace=False)
idx_atk=np.where(lab.values!=nv)[0]
keep=np.sort(np.concatenate([idx_norm,idx_atk])); ts=ts.iloc[keep].reset_index(drop=True)
fams=[v for v in ts["Label"].unique() if v!=nv]
print("after subsample:",len(ts),"| families:",fams); print(ts["Label"].value_counts().to_string())

device: cuda
loaded ts flows: 92131
after subsample: 53542 | families: ['DoS', 'DDoS', 'Blackhole', 'Wormhole', 'Replay']
Label
Benign       20000
Blackhole    12116
Wormhole     10862
DDoS          7316
Replay        2025
DoS           1223


In [15]:
# Build padded 3-channel tensor (size, inter-arrival, direction) once, standardized on valid packets.
def to_channels(pt,ps,pdir,L):
    pt=np.asarray(pt,dtype=np.float64); ps=np.asarray(ps,dtype=np.float64); pdir=np.asarray(pdir,dtype=np.float64)
    n=min(len(pt),L); arr=np.zeros((L,3),dtype=np.float32); m=np.zeros(L,dtype=np.float32)
    if n>0:
        iat=np.zeros(n)
        if n>1: iat[1:]=np.clip(np.diff(pt[:n]),0,None)
        arr[:n,0]=np.log1p(ps[:n]); arr[:n,1]=np.log1p(iat); arr[:n,2]=pdir[:n]; m[:n]=1.0
    return arr,m
N=len(ts); X=np.zeros((N,L,3),dtype=np.float32); M=np.zeros((N,L),dtype=np.float32)
pt_c,ps_c,pd_c=ts["packet_time"].values,ts["packet_size"].values,ts["packet_dir"].values
for i in range(N):
    X[i],M[i]=to_channels(pt_c[i],ps_c[i],pd_c[i],L)
    if (i+1)%20000==0: print("  built",i+1)
y=(ts["Label"].values!=nv).astype(np.int64); fam=ts["Label"].values
vb=M.astype(bool)
for ch in range(3):
    vals=X[:,:,ch][vb]; mu,sd=float(vals.mean()),float(vals.std())+1e-6
    X[:,:,ch]=(X[:,:,ch]-mu)/sd; X[:,:,ch][~vb]=0.0
X=np.transpose(X,(0,2,1)).copy(); print("tensor:",X.shape,"mem %.0f MB"%(X.nbytes/1e6)); del pt_c,ps_c,pd_c,M,vb; gc.collect()

  built 20000
  built 40000
tensor: (53542, 3, 1105) mem 710 MB


30

In [16]:
# 1D-CNN with strided pooling + balanced sampler (normal and each attack family equally weighted per batch)
class PacketCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv1d(3,32,7,stride=2,padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32,64,5,stride=2,padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64,96,5,stride=2,padding=2), nn.BatchNorm1d(96), nn.ReLU(),
            nn.Conv1d(96,96,3,stride=2,padding=1), nn.BatchNorm1d(96), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.head=nn.Sequential(nn.Linear(96,48), nn.ReLU(), nn.Dropout(0.2), nn.Linear(48,1))
    def forward(self,x): return self.head(self.net(x).squeeze(-1)).squeeze(-1)

def balanced_weights(fam_tr):
    fam_tr=np.asarray(fam_tr); seen=[f for f in np.unique(fam_tr) if f!=nv]
    w=np.zeros(len(fam_tr),dtype=np.float64); isn=fam_tr==nv
    w[isn]=0.5/max(isn.sum(),1)
    for f in seen:
        m=fam_tr==f; w[m]=0.5/(len(seen)*max(m.sum(),1))
    return w

def split_idx(fam,held_out,seed):
    seen=[f for f in fams if f!=held_out]
    def sp(ix,fr,sd):
        ix=np.array(ix); r=np.random.default_rng(sd); r.shuffle(ix); out={}; s=0; ks=list(fr)
        for i,k in enumerate(ks):
            e=len(ix) if i==len(ks)-1 else s+int(round(fr[k]*len(ix))); out[k]=ix[s:e]; s=e
        return out
    tr=[];ca=[];se=[];sh=[]
    ns=sp(np.where(fam==nv)[0],CFG["normal_fracs"],seed); tr+=list(ns["train"]);ca+=list(ns["cal"]);se+=list(ns["test_seen"]);sh+=list(ns["test_shift"])
    for j,f in enumerate(seen):
        fs=sp(np.where(fam==f)[0],CFG["family_fracs"],seed+j+1); tr+=list(fs["train"]);ca+=list(fs["cal"]);se+=list(fs["test_seen"])
    sh+=list(np.where(fam==held_out)[0])
    return (np.array(sorted(a)) for a in (tr,ca,se,sh))

def train_probs(tr,ca,se,sh,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Xt=torch.tensor(X[tr]); yt=torch.tensor(y[tr],dtype=torch.float32)
    wts=torch.tensor(balanced_weights(fam[tr]),dtype=torch.double)
    sampler=WeightedRandomSampler(wts,num_samples=len(tr),replacement=True)
    dl=DataLoader(TensorDataset(Xt,yt),batch_size=CFG["batch"],sampler=sampler,drop_last=True)
    net=PacketCNN().to(dev); opt=torch.optim.Adam(net.parameters(),lr=CFG["lr"],weight_decay=CFG["weight_decay"])
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CFG["epochs"]); lossf=nn.BCEWithLogitsLoss()
    net.train()
    for ep in range(CFG["epochs"]):
        for xb,yb in dl:
            xb,yb=xb.to(dev),yb.to(dev); opt.zero_grad(); lossf(net(xb),yb).backward(); opt.step()
        sched.step()
    net.eval()
    def prob(ix):
        out=[]
        with torch.no_grad():
            for s in range(0,len(ix),512):
                out.append(torch.sigmoid(net(torch.tensor(X[ix[s:s+512]]).to(dev))).cpu().numpy())
        return np.concatenate(out) if len(ix) else np.array([])
    p=(prob(ca),prob(se),prob(sh)); del net,Xt,yt; gc.collect()
    if dev=="cuda": torch.cuda.empty_cache()
    return p

In [ ]:
# Held-out protocol on the SEQUENCE representation -> conformal coverage panel (10 seeds)
def cov_detail(p,yv,qh):
    p=np.asarray(p);yv=np.asarray(yv);ia=(1-p)<=qh;ino=p<=qh; inset=np.where(yv==1,ia,ino)
    return float(inset.mean()), float(np.mean([inset[yv==k].mean() for k in np.unique(yv)])), float((ia.astype(int)+ino.astype(int)).mean())
rows=[]
for F in fams:
    for seed in CFG["seeds"]:
        tr,ca,se,sh=split_idx(fam,F,seed)
        p_ca,p_se,p_sh=train_probs(tr,ca,se,sh,seed)
        qh=conformal_qhat(p_ca,y[ca],alpha=CFG["alpha"])
        m_sh,b_sh,sz=cov_detail(p_sh,y[sh],qh); m_se,_,_=cov_detail(p_se,y[se],qh)
        rows.append({"held_out":str(F),"seed":seed,"shift_balacc":balanced_accuracy_score(y[sh],(p_sh>=.5).astype(int)),
            "shift_cov_marg":m_sh,"shift_cov_bal":b_sh,"shift_setsize":sz,"seen_cov_marg":m_se})
        print(f"  seq {F} seed{seed}: balacc={rows[-1]['shift_balacc']:.3f} cov_marg={m_sh:.3f} cov_bal={b_sh:.3f}")
    print(" SEQUENCE",F,"done")
Q=pd.DataFrame(rows); Q.to_csv(os.path.join(CFG["report_dir"],"16_uavcas_ts_coverage_raw.csv"),index=False)
print("\nin-distribution (seen) marginal coverage, sequence model: %.3f +/- %.3f"%(Q["seen_cov_marg"].mean(),Q["seen_cov_marg"].std()))

  seq DoS seed0: balacc=0.972 cov_marg=0.988 cov_bal=0.990
  seq DoS seed1: balacc=0.977 cov_marg=0.989 cov_bal=0.991
  seq DoS seed2: balacc=0.959 cov_marg=0.988 cov_bal=0.990
  seq DoS seed3: balacc=0.973 cov_marg=0.985 cov_bal=0.988
  seq DoS seed4: balacc=0.977 cov_marg=0.988 cov_bal=0.990
  seq DoS seed5: balacc=0.963 cov_marg=0.988 cov_bal=0.990
  seq DoS seed6: balacc=0.958 cov_marg=0.998 cov_bal=0.998
  seq DoS seed7: balacc=0.962 cov_marg=0.992 cov_bal=0.994
  seq DoS seed8: balacc=0.964 cov_marg=0.993 cov_bal=0.995
  seq DoS seed9: balacc=0.974 cov_marg=0.991 cov_bal=0.992
 SEQUENCE DoS done
  seq DDoS seed0: balacc=0.968 cov_marg=0.997 cov_bal=0.993
  seq DDoS seed1: balacc=0.969 cov_marg=0.996 cov_bal=0.992
  seq DDoS seed2: balacc=0.961 cov_marg=0.996 cov_bal=0.992
  seq DDoS seed3: balacc=0.959 cov_marg=0.995 cov_bal=0.988
  seq DDoS seed4: balacc=0.973 cov_marg=0.998 cov_bal=0.995
  seq DDoS seed5: balacc=0.967 cov_marg=0.996 cov_bal=0.990
  seq DDoS seed6: balacc=0.964 

In [ ]:
# Sequence panel with std (10 seeds), and side-by-side vs tabular on the SAME flows
ms=lambda g,col:"%.3f \u00b1 %.3f"%(g[col].mean(),g[col].std())
seqtab=pd.DataFrame([{"held_out":F,"seq_balacc":ms(g,"shift_balacc"),"seq_cov_marg":ms(g,"shift_cov_marg"),"seq_cov_bal":ms(g,"shift_cov_bal")} for F,g in Q.groupby("held_out")])
print("=== UAV-CAS sequence-model coverage panel (mean +/- std over 10 seeds) ==="); print(seqtab.to_string(index=False))
seqtab.to_csv(os.path.join(CFG["report_dir"],"16_uavcas_ts_panel.csv"),index=False)
seqm=Q.groupby("held_out").agg(seq_balacc=("shift_balacc","mean"),seq_cov_marg=("shift_cov_marg","mean"),seq_cov_std=("shift_cov_marg","std")).round(3)
try:
    tab=pd.read_csv(CAS_STAT_CMP).groupby("held_out").agg(tab_balacc=("shift_balacc","mean"),tab_cov_marg=("shift_cov_marg","mean")).round(3)
    comp=tab.join(seqm)
except Exception as e:
    print("tabular csv missing",e); comp=seqm
print("\n=== Coverage under two representations of the SAME flows (target 0.90) ==="); print(comp.to_string())
comp.to_csv(os.path.join(CFG["report_dir"],"16_uavcas_representation_compare.csv"))
if "tab_cov_marg" in comp and comp["tab_cov_marg"].notna().all():
    print("\nSpearman(tabular, sequence) coverage across families = %.3f"%spearmanr(comp["tab_cov_marg"],comp["seq_cov_marg"]).correlation)
    print("families below 0.80 -- tabular %d/5, sequence %d/5"%((comp["tab_cov_marg"]<0.8).sum(),(comp["seq_cov_marg"]<0.8).sum()))

In [ ]:
# FIGURE: tabular vs sequence marginal coverage per family (with sequence error bars)
import matplotlib.pyplot as plt
o=comp.reset_index(); x=np.arange(len(o)); w=0.38
fig,ax=plt.subplots(figsize=(9,4.2))
if "tab_cov_marg" in o: ax.bar(x-w/2,o["tab_cov_marg"],w,label="tabular (47 features, XGBoost)",color="#264653")
yerr=o["seq_cov_std"] if "seq_cov_std" in o else None
ax.bar(x+w/2,o["seq_cov_marg"],w,yerr=yerr,capsize=3,label="sequence (raw packets, 1D-CNN)",color="#e76f51")
ax.axhline(0.90,ls="--",color="gray"); ax.set_xticks(x); ax.set_xticklabels(o["held_out"],rotation=20,ha="right")
ax.set_ylabel("shift marginal coverage"); ax.set_ylim(0,1.05)
ax.set_title("Coverage collapse is representation-invariant: same flows, two representations"); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(CFG["fig_dir"],"16_representation_invariance.png"),dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
!git add reports/ figures/ notebooks/
!git commit -m "16 representation-invariance (refined): 1D-CNN on raw packet sequences, 10 seeds, balanced sampling, longer schedule; conformal coverage vs tabular"
!git push origin main